# FastAPI Summarization Service Testing

This notebook tests the FastAPI text summarization service to verify that summaries are generated correctly and stop at the right moment.

## Setup

Make sure the FastAPI server is running:
```bash
cd fastapi
python main.py
```

In [1]:
import requests
import json
from typing import Dict
import pandas as pd

In [2]:
# API configuration
API_BASE_URL = "http://localhost:8000"
SUMMARIZE_ENDPOINT = f"{API_BASE_URL}/api/v1/summarize"
BATCH_SUMMARIZE_ENDPOINT = f"{API_BASE_URL}/api/v1/batch-summarize"

## 1. Health Check

In [3]:
# Check if API is running
try:
    response = requests.get(f"{API_BASE_URL}/health")
    print(f"API Status: {response.json()}")
except Exception as e:
    print(f"Error: {e}")
    print("Make sure the FastAPI server is running!")

API Status: {'status': 'healthy'}


## 2. Test Single Text Summarization

In [4]:
# Test text samples
test_texts = [
    """
    Президент России Владимир Путин провел встречу с главами регионов.
    На встрече обсуждались вопросы социально-экономического развития субъектов федерации.
    Особое внимание было уделено реализации национальных проектов и поддержке малого бизнеса.
    Глава государства подчеркнул важность повышения качества жизни граждан и создания
    комфортной городской среды. Также были рассмотрены вопросы развития инфраструктуры
    и цифровизации государственных услуг.
    """,
    """
    Московский метрополитен открыл новую станцию на Большой кольцевой линии.
    Станция оборудована современными системами безопасности и навигации.
    Пассажиры отмечают удобство новой станции и улучшение транспортной доступности района.
    Строительство станции велось в течение трех лет с применением передовых технологий.
    """,
    """
    Российские ученые разработали новый метод лечения онкологических заболеваний.
    Клинические испытания показали высокую эффективность препарата.
    Новая методика позволяет значительно увеличить выживаемость пациентов.
    Препарат уже прошел первую фазу клинических испытаний и показал обнадеживающие результаты.
    Ученые планируют начать массовое производство в следующем году.
    """
]

In [5]:
def summarize_single(text: str) -> Dict:
    """Summarize a single text using the API."""
    response = requests.post(
        SUMMARIZE_ENDPOINT,
        json={"text": text.strip()}
    )
    response.raise_for_status()
    return response.json()

# Test single summarization
print("Testing Single Text Summarization\n" + "="*60)
for i, text in enumerate(test_texts, 1):
    print(f"\n📝 Test {i}:")
    print(f"Original text ({len(text)} chars):")
    print(text.strip()[:200] + "..." if len(text) > 200 else text.strip())

    result = summarize_single(text)
    summary = result['summary']

    print(f"\n✅ Summary ({len(summary)} chars):")
    print(summary)
    print(f"\n📊 Compression ratio: {len(summary)/len(text):.2%}")
    print(f"Model used: {result['model_used']}")
    print("-" * 60)

Testing Single Text Summarization

📝 Test 1:
Original text (475 chars):
Президент России Владимир Путин провел встречу с главами регионов.
    На встрече обсуждались вопросы социально-экономического развития субъектов федерации.
    Особое внимание было уделено реализации...

✅ Summary (323 chars):
Президент России Владимир Путин провел встречу с главами регионов. На встрече обсуждались вопросы социально-экономического развития субъектов федерации, реализации национальных проектов и поддержке малого бизнеса. Глава государства подчеркнул важность повышения качества жизни граждан и создания комфортной городской среды.

📊 Compression ratio: 68.00%
Model used: IlyaGusev/rut5_base_sum_gazeta
------------------------------------------------------------

📝 Test 2:
Original text (334 chars):
Московский метрополитен открыл новую станцию на Большой кольцевой линии.
    Станция оборудована современными системами безопасности и навигации.
    Пассажиры отмечают удобство новой станции и улучше...



## 3. Analyze Summary Quality

In [7]:
# Create a comparison dataframe
comparison_data = []

for i, text in enumerate(test_texts, 1):
    result = summarize_single(text)
    summary = result['summary']

    comparison_data.append({
        'Test #': i,
        'Original Length': len(text.strip()),
        'Summary Length': len(summary),
        'Compression Ratio': f"{len(summary)/len(text):.2%}",
        'Word Count (Original)': len(text.split()),
        'Word Count (Summary)': len(summary.split()),
        'Summary': summary[:100] + '...' if len(summary) > 100 else summary
    })

df = pd.DataFrame(comparison_data)
print("\n📊 Summary Statistics:")
print("="*80)
display(df)


📊 Summary Statistics:


,Test #,Original Length,Summary Length,Compression Ratio,Word Count (Original),Word Count (Summary),Summary
0,1,465,323,68.00%,51,37,Президент России Владимир Путин провел встречу...
1,2,324,228,68.26%,37,26,Московский метрополитен открыл новую станцию н...
2,3,383,309,78.63%,40,35,Российские ученые разработали новый метод лече...


## 4. Test Edge Cases

In [8]:
# Test with very short text
short_text = "Москва - столица России."
print("Testing with very short text:")
print(f"Original: {short_text}")
result = summarize_single(short_text)
print(f"Summary: {result['summary']}")
print()

Testing with very short text:
Original: Москва - столица России.
Summary: Москву - столицу России. Москва - столица России. Москва — столица и столица РФ.



In [9]:
# Test with longer text
long_text = """
В Санкт-Петербурге прошел международный экономический форум, который собрал более 10 тысяч участников
из 140 стран мира. На форуме обсуждались ключевые вопросы развития мировой экономики, цифровой
трансформации бизнеса и устойчивого развития. Особое внимание было уделено вопросам искусственного
интеллекта, блокчейн-технологий и развития финтех-индустрии. Участники форума отметили важность
международного сотрудничества в условиях глобальных вызовов. Были подписаны соглашения о сотрудничестве
на общую сумму более 50 миллиардов рублей. Эксперты прогнозируют, что реализация достигнутых договоренностей
позволит создать тысячи новых рабочих мест и ускорить технологическое развитие регионов.
"""

print("Testing with longer text:")
print(f"Original length: {len(long_text)} chars, {len(long_text.split())} words")
result = summarize_single(long_text)
summary = result['summary']
print(f"\nSummary length: {len(summary)} chars, {len(summary.split())} words")
print(f"\nSummary: {summary}")
print(f"\nCompression ratio: {len(summary)/len(long_text):.2%}")

Testing with longer text:
Original length: 695 chars, 80 words

Summary length: 338 chars, 37 words

Summary: В Санкт-Петербурге прошел международный экономический форум, на котором обсуждались ключевые вопросы развития мировой экономики, цифровой трансформации бизнеса и устойчивого развития. Эксперты прогнозируют, что реализация достигнутых договоренностей позволит создать тысячи новых рабочих мест и ускорить технологическое развитие регионов.

Compression ratio: 48.63%


## 5. Check for Generation Issues

In [10]:
# Check if summaries have proper endings (no truncation or repetition)
print("Checking summary quality:\n" + "="*60)

for i, text in enumerate(test_texts, 1):
    result = summarize_single(text)
    summary = result['summary']

    # Check for repetition
    words = summary.split()
    has_repetition = False
    for j in range(len(words) - 3):
        if words[j:j+3] == words[j+3:j+6]:
            has_repetition = True
            break

    # Check if ends properly
    ends_properly = summary.strip()[-1] in '.!?'

    print(f"\nTest {i}:")
    print(f"Summary: {summary}")
    print(f"✓ Ends properly: {ends_properly}")
    print(f"✓ No repetition: {not has_repetition}")
    print(f"✓ Length: {len(summary)} chars, {len(summary.split())} words")
    print("-" * 60)

Checking summary quality:

Test 1:
Summary: Президент России Владимир Путин провел встречу с главами регионов. На встрече обсуждались вопросы социально-экономического развития субъектов федерации, реализации национальных проектов и поддержке малого бизнеса. Глава государства подчеркнул важность повышения качества жизни граждан и создания комфортной городской среды.
✓ Ends properly: True
✓ No repetition: True
✓ Length: 323 chars, 37 words
------------------------------------------------------------

Test 2:
Summary: Московский метрополитен открыл новую станцию на Большой кольцевой линии. Станция оборудована современными системами безопасности и навигации. Пассажиры отмечают удобство новой станции и улучшение транспортной доступности района.
✓ Ends properly: True
✓ No repetition: True
✓ Length: 228 chars, 26 words
------------------------------------------------------------

Test 3:
Summary: Российские ученые разработали новый метод лечения онкологических заболеваний. Новая методика позв

## Summary

This notebook tests the FastAPI summarization service with:
- ✅ Single text summarization
- ✅ Edge cases (short and long texts)
- ✅ Quality checks (proper endings, no repetition)

The new generation parameters (`max_new_tokens`, `early_stopping`, etc.) should prevent the model from generating endless text.